# Project 1: emotion classification on ISEAR

Comparing 3 representations × 3 architectures with cross-validation and a Bayesian ROPE analysis, reusing the tutorial code in `part1/`.

| Section | Who |
| --- | --- |
| 1. Data, 2. Folds, 3. Tokeniser | Britt |
| 4. Representations | Zeno |
| 5. Architectures | Armani |
| 6. Experiments, 7. Bayesian comparison, 8. Error analysis | Together |

Run `python explore_data.py` once first: it creates `data/isear_clean.csv`.

In [2]:
from pathlib import Path

import numpy as np
import pandas as pd

import part1
from part1.clslib import NamedClasses

results_folder = Path("results/project1")
results_folder.mkdir(parents=True, exist_ok=True)

## 1. Data
Cleaned ISEAR (see `explore_data.py`): 7,516 raw texts, 84 removed (10 placeholders, 21 same text with conflicting labels, 53 duplicates), 7,432 left. The classes are balanced, so accuracy is a fair metric (majority baseline 14.5%).

In [3]:
LABELS = ["joy", "fear", "anger", "sadness", "disgust", "shame", "guilt"]
outcome_space = NamedClasses(LABELS)

df = pd.read_csv("data/isear_clean.csv")
pairs = list(zip(df["text"], df["label_id"]))  # (text, label) pairs, as in the tutorial
print(len(pairs), "texts")
df["label"].value_counts()

7432 texts


label
joy        1079
anger      1077
fear       1070
disgust    1060
shame      1053
sadness    1050
guilt      1043
Name: count, dtype: int64

## 2. Cross-validation folds
5 folds, repeated with 3 seeds (15 accuracies per system), using the tutorial's `cross_validation_split`. Every system uses these same folds, so the scores are paired for the Bayesian comparison.

In [4]:
seeds = [0, 3, 5]
num_folds = 5
cv_sets = [part1.experiment.cross_validation_split(pairs, k=num_folds, seed=seed) for seed in seeds]

# check: are the classes spread evenly over the folds?
for fold, split in cv_sets[0].items():
    counts = pd.Series([label for _, label in split["validation"]]).value_counts().sort_index()
    print(f"fold {fold}: {len(split['validation'])} texts, per class {counts.tolist()}")

fold 0: 1487 texts, per class [231, 206, 186, 239, 219, 210, 196]
fold 1: 1487 texts, per class [198, 217, 227, 199, 215, 208, 223]
fold 2: 1486 texts, per class [237, 186, 228, 202, 221, 219, 193]
fold 3: 1486 texts, per class [207, 221, 219, 207, 206, 216, 210]
fold 4: 1486 texts, per class [206, 240, 217, 203, 199, 200, 221]


## 3. Tokeniser
BPE tokeniser trained with the tutorial's `train_tokenizer` on the ISEAR texts themselves (~165k words is enough for a small vocabulary). It uses no labels, so training on all texts does not leak label information (the tutorial does the same).

To choose the vocabulary size, we compare how finely each size splits words: a good size keeps frequent words, such as emotion words, whole without making the vocabulary huge.

In [5]:
texts = [text for text, _ in pairs]
n_words = sum(len(t.split()) for t in texts)
example = "I felt ashamed when I shouted at my girlfriend."

for size in [500, 1000, 2000, 5000]:
    tok = part1.experiment.train_tokenizer(texts, size, min_frequency=2)
    n_tokens = sum(len(tok.encode(t).ids) for t in texts)
    print(f"{size:>5}: {n_tokens / n_words:.2f} tokens per word | {tok.encode(example).tokens}")




  500: 2.10 tokens per word | ['▁I', '▁felt', '▁as', 'ham', 'ed', '▁when', '▁I', '▁sh', 'ou', 'ted', '▁at', '▁my', '▁girl', 'friend', '.']



 1000: 1.74 tokens per word | ['▁I', '▁felt', '▁ashamed', '▁when', '▁I', '▁sh', 'ou', 'ted', '▁at', '▁my', '▁girlfriend', '.']



 2000: 1.50 tokens per word | ['▁I', '▁felt', '▁ashamed', '▁when', '▁I', '▁sh', 'ou', 'ted', '▁at', '▁my', '▁girlfriend.']



 5000: 1.28 tokens per word | ['▁I', '▁felt', '▁ashamed', '▁when', '▁I', '▁shouted', '▁at', '▁my', '▁girlfriend.']


In [6]:
vocab_size = 1000  # same as tutorial (500 is too small and with 2000 puctuation gets glued on the words)
tokenizer = part1.experiment.train_tokenizer(texts, vocab_size, min_frequency=2)
tokenizer.save(f"{results_folder}/bpe_vs{vocab_size}.json")
print(tokenizer.encode(example).tokens)




['▁I', '▁felt', '▁ashamed', '▁when', '▁I', '▁sh', 'ou', 'ted', '▁at', '▁my', '▁girlfriend', '.']


## 4. Representations (Zeno)
- token indicators (tutorial)
- bag-of-words (tutorial exercise)
- one extra of our choice

In [7]:
from collections import Counter
from tokenizers import Tokenizer

class Encoder:

    def __len__(self) -> int:
        """Dimensionality of the representation space"""
        raise NotImplementedError("Concrete implementations will be discussed later")

    def __call__(self, x: str) -> 'np.array':
        """Represent x as a D-dimensional vector of real numbers"""
        raise NotImplementedError("Concrete implementations will be discussed later")


class TokenIndicator(Encoder):
    """h_v(x) = 1 if BPE token v occurs in x, else 0 (tutorial)."""

    def __init__(self, tokenizer: Tokenizer):
        self.tokenizer = tokenizer
        
    def __len__(self) -> int:
        """Dimensionality of the representation space"""
        return len(tokenizer.get_vocab())

    def __call__(self, x: str) -> 'np.array':
        """Represent x as a D-dimensional vector of real numbers"""
        unique_token_ids = list(set(self.tokenizer.encode(x).ids))

        # Set the 0 vector to 1 in every word that appears
        vector = np.zeros(len(self))
        vector[unique_token_ids] = 1
        return vector


class BagOfWords(Encoder):
    """h_v(x) = number of times BPE token v occurs in x (tutorial exercise)."""

    def __init__(self, tokenizer: Tokenizer):
        self.tokenizer = tokenizer
        self.dim = tokenizer.get_vocab_size()

    def __len__(self) -> int:
        return self.dim

    def __call__(self, x: str) -> np.ndarray:
        vector = np.zeros(self.dim)
        # np.add.at accumulates repeated ids; vector[ids] += 1 would count each id only once
        np.add.at(vector, self.tokenizer.encode(x).ids, 1)
        return vector


class BagOfUnigramsAndBigrams(Encoder):
    """
    Counts of BPE tokens plus counts of adjacent token pairs (the tutorial's 'bag of bigrams').
    Only bigrams seen at least `min_count` times in `texts` get a dimension, otherwise D ~ V^2.
    """

    def __init__(self, tokenizer: Tokenizer, texts: list, min_count: int = 2):
        self.tokenizer = tokenizer
        self.num_unigrams = tokenizer.get_vocab_size()
        bigram_counts = Counter(
            pair
            for text in texts
            for pair in self._bigrams(tokenizer.encode(text).ids)
        )
        kept = sorted(pair for pair, count in bigram_counts.items() if count >= min_count)
        # bigram dimensions come after the V unigram dimensions
        self.bigram_index = {pair: self.num_unigrams + i for i, pair in enumerate(kept)}
        self.dim = self.num_unigrams + len(self.bigram_index)

    @staticmethod
    def _bigrams(ids):
        return zip(ids, ids[1:])

    def __len__(self) -> int:
        return self.dim

    def __call__(self, x: str) -> np.ndarray:
            ids = self.tokenizer.encode(x).ids
            vector = np.zeros(self.dim)
            np.add.at(vector, ids, 1)
            bigram_dims = [self.bigram_index[p] for p in self._bigrams(ids) if p in self.bigram_index]
            np.add.at(vector, bigram_dims, 1)  # unseen bigrams are simply dropped
            return vector
    
    """FOR SECTION 8!
    def coords(self):
        Readable axis labels, e.g. for inspecting learned weights in the error analysis.

        id_to_token = {i: t for t, i in self.tokenizer.get_vocab().items()}
        labels = [id_to_token[i] for i in range(self.num_unigrams)]
        labels += [id_to_token[a] + " " + id_to_token[b] for (a, b) in self.bigram_index]
        return labels
    """
    

## 5. Architectures (Armani)
Log-linear and 1-hidden-layer MLP, plus one extra architecture

In [8]:
class LogLinear:
    def __init__(self, encoder, K=7):
        self.encoder = encoder
        self.K = K
        self.W = None

    def probabilities(self, x):
        s = self.W @ x
        e = np.exp(s - s.max())
        return e / e.sum()

    def train(self, data, lr=0.01, reg_strength=1e-3, num_epochs=200):
        x_and_y = [(self.encoder(text), y) for text, y in data]
        self.W = np.zeros((self.K, len(self.encoder)))

        for epoch in range(num_epochs):
            print(f"Epoch {epoch+1}")
            grad = np.zeros_like(self.W)
            for x, y in x_and_y:
                grad[y] += x
                grad -= np.outer(self.probabilities(x), x)
            self.W += lr * (grad / len(data) - reg_strength * self.W)

    def predict(self, text):

        pred_int = int(np.argmax(self.W @ self.encoder(text)))
        return LABELS[pred_int]

model_loglinear = LogLinear(BagOfWords(tokenizer))
model_loglinear.train(cv_sets[0][0]["training"])

Epoch 1
Epoch 2
Epoch 3
Epoch 4
Epoch 5
Epoch 6
Epoch 7
Epoch 8
Epoch 9
Epoch 10
Epoch 11
Epoch 12
Epoch 13
Epoch 14
Epoch 15
Epoch 16
Epoch 17
Epoch 18
Epoch 19
Epoch 20
Epoch 21
Epoch 22
Epoch 23
Epoch 24
Epoch 25
Epoch 26
Epoch 27
Epoch 28
Epoch 29
Epoch 30
Epoch 31
Epoch 32
Epoch 33
Epoch 34
Epoch 35
Epoch 36
Epoch 37
Epoch 38
Epoch 39
Epoch 40
Epoch 41
Epoch 42
Epoch 43
Epoch 44
Epoch 45
Epoch 46
Epoch 47
Epoch 48
Epoch 49
Epoch 50
Epoch 51
Epoch 52
Epoch 53
Epoch 54
Epoch 55
Epoch 56
Epoch 57
Epoch 58
Epoch 59
Epoch 60
Epoch 61
Epoch 62
Epoch 63
Epoch 64
Epoch 65
Epoch 66
Epoch 67
Epoch 68
Epoch 69
Epoch 70
Epoch 71
Epoch 72
Epoch 73
Epoch 74
Epoch 75
Epoch 76
Epoch 77
Epoch 78
Epoch 79
Epoch 80
Epoch 81
Epoch 82
Epoch 83
Epoch 84
Epoch 85
Epoch 86
Epoch 87
Epoch 88
Epoch 89
Epoch 90
Epoch 91
Epoch 92
Epoch 93
Epoch 94
Epoch 95
Epoch 96
Epoch 97
Epoch 98
Epoch 99
Epoch 100
Epoch 101
Epoch 102
Epoch 103
Epoch 104
Epoch 105
Epoch 106
Epoch 107
Epoch 108
Epoch 109
Epoch 110
Epoch 11

In [9]:
model_loglinear.predict("very happy")

'joy'

In [10]:
# Voor een 10!
class MLP:
    def __init__(self, encoder, H=100, K=7, seed=42):
        self.encoder = encoder
        self.H = H
        self.K = K
        self.rng = np.random.default_rng(seed)

    def softmax(self, s):
        e = np.exp(s - s.max())
        return e / e.sum()

    def forward(self, x):
        z = self.W1 @ x + self.b1
        h = np.maximum(z, 0) # ReLU
        p = self.softmax(self.W2 @ h + self.b2)
        return z, h, p # z and h needed for bwd pass

    def train(self, data, lr=0.01, reg_strength=1e-3, num_epochs=200):
        x_and_y = [(self.encoder(text), y) for text, y in data]
        d = len(self.encoder)

        # small rand init, but not 0
        self.W1 = self.rng.normal(size=(self.H, d)) * 0.01
        self.b1 = np.zeros(self.H)
        self.W2 = self.rng.normal(size=(self.K, self.H)) * 0.01
        self.b2 = np.zeros(self.K)

        for epoch in range(num_epochs):
            print(f"Epoch {epoch+1}")
            gW1, gb1 = np.zeros_like(self.W1), np.zeros_like(self.b1)
            gW2, gb2 = np.zeros_like(self.W2), np.zeros_like(self.b2)

            for x, y in x_and_y:
                z, h, p = self.forward(x)
                ds = p.copy()
                ds[y] -= 1
                gW2 += np.outer(ds, h)
                gb2 += ds
                dz = (self.W2.T @ ds) * (z > 0)
                gW1 += np.outer(dz, x)
                gb1 += dz

            n = len(data)
            self.W1 -= lr * (gW1 / n + reg_strength * self.W1)
            self.b1 -= lr * gb1 / n
            self.W2 -= lr * (gW2 / n + reg_strength * self.W2)
            self.b2 -= lr * gb2 / n

    def predict(self, text):
        _, _, p = self.forward(self.encoder(text))
        pred_int = int(np.argmax(p))
        return LABELS[pred_int]

model_mlp = MLP(BagOfWords(tokenizer))
model_mlp.train(cv_sets[0][0]["training"])

Epoch 1
Epoch 2
Epoch 3
Epoch 4
Epoch 5
Epoch 6
Epoch 7
Epoch 8
Epoch 9
Epoch 10
Epoch 11
Epoch 12
Epoch 13
Epoch 14
Epoch 15
Epoch 16
Epoch 17
Epoch 18
Epoch 19
Epoch 20
Epoch 21
Epoch 22
Epoch 23
Epoch 24
Epoch 25
Epoch 26
Epoch 27
Epoch 28
Epoch 29
Epoch 30
Epoch 31
Epoch 32
Epoch 33
Epoch 34
Epoch 35
Epoch 36
Epoch 37
Epoch 38
Epoch 39
Epoch 40
Epoch 41
Epoch 42
Epoch 43
Epoch 44
Epoch 45
Epoch 46
Epoch 47
Epoch 48
Epoch 49
Epoch 50
Epoch 51
Epoch 52
Epoch 53
Epoch 54
Epoch 55
Epoch 56
Epoch 57
Epoch 58
Epoch 59
Epoch 60
Epoch 61
Epoch 62
Epoch 63
Epoch 64
Epoch 65
Epoch 66
Epoch 67
Epoch 68
Epoch 69
Epoch 70
Epoch 71
Epoch 72
Epoch 73
Epoch 74
Epoch 75
Epoch 76
Epoch 77
Epoch 78
Epoch 79
Epoch 80
Epoch 81
Epoch 82
Epoch 83
Epoch 84
Epoch 85
Epoch 86
Epoch 87
Epoch 88
Epoch 89
Epoch 90
Epoch 91
Epoch 92
Epoch 93
Epoch 94
Epoch 95
Epoch 96
Epoch 97
Epoch 98
Epoch 99
Epoch 100
Epoch 101
Epoch 102
Epoch 103
Epoch 104
Epoch 105
Epoch 106
Epoch 107
Epoch 108
Epoch 109
Epoch 110
Epoch 11

In [11]:
model_mlp.predict("very happy")

'anger'

In [ ]:
# MLP
class MLP_hella_layers:
    def __init__(self, encoder, hidden=(100,), K=7, seed=42):
        self.encoder = encoder
        self.hidden = list(hidden)
        self.K = K
        self.rng = np.random.default_rng(seed)

    def softmax(self, s):
        e = np.exp(s - s.max())
        return e / e.sum()

    def forward(self, x):
        hs, zs = [x], []
        L = len(self.Ws)
        for l in range(L):
            z = self.Ws[l] @ hs[-1] + self.bs[l]
            zs.append(z)
            if l < L - 1:
                hs.append(np.maximum(z, 0))
        return zs, hs, self.softmax(zs[-1])

    def train(self, data, lr=0.01, reg_strength=1e-3, num_epochs=20):
        x_and_y = [(self.encoder(text), y) for text, y in data]
        sizes = [len(self.encoder)] + self.hidden + [self.K]

        self.Ws = [self.rng.normal(size=(sizes[i+1], sizes[i])) * np.sqrt(2 / sizes[i]) for i in range(len(sizes) - 1)]
        self.bs = [np.zeros(sizes[i+1]) for i in range(len(sizes) - 1)]
        L, n = len(self.Ws), len(data)

        for epoch in range(num_epochs):
            print(f"Epoch {epoch+1}/{num_epochs}")

            gWs = [np.zeros_like(W) for W in self.Ws]
            gbs = [np.zeros_like(b) for b in self.bs]

            for x, y in x_and_y:
                zs, hs, p = self.forward(x)
                delta = p.copy()
                delta[y] -= 1
                for l in reversed(range(L)):
                    gWs[l] += np.outer(delta, hs[l])
                    gbs[l] += delta
                    if l > 0:
                        delta = (self.Ws[l].T @ delta) * (zs[l-1] > 0)
                for l in range(L):
                    self.Ws[l] -= lr * (gWs[l] / n + reg_strength * self.Ws[l])
                    self.bs[l] -= lr * gbs[l] / n

    def predict(self, text):
        _, _, p = self.forward(self.encoder(text))
        return LABELS[int(np.argmax(p))]

blaze_it_array = (67,) * 420
model_mlp_hella_layers = MLP_hella_layers(BagOfWords(tokenizer), hidden=blaze_it_array)
model_mlp_hella_layers.train(cv_sets[0][0]["training"])

Epoch 1/20
Epoch 2/20
Epoch 3/20
Epoch 4/20
Epoch 5/20


## 6. Experiments (together)


## 7. Bayesian comparison with a ROPE (together)
Use tutorial 

## 8. Error analysis (together)
per-class F1 and a confusion matrix